Environment note
----------------
All training and dataset generation for this project ran under WSL2
(Ubuntu 22.04) with GPU acceleration. Paths in those notebooks point
at /home/admins/ and reflect that environment.

Phase 4 onward runs on Windows, CPU only - inference and demo work,
no training. Paths here point at D:\lip_codebase_clean\.

The two environments were verified equivalent: the same model on the
same test set reproduces 0.7150 (full test) and 0.7556 (real
recordings only) to four decimal places on both.

One difference: albumentations does not install on Windows Python 3.11
(its stringzilla dependency needs a C++ compiler), so dataset
generation is WSL-only. The datasets are already built.

See docs/environment/ for exact package versions on both platforms.

In [ ]:
import cv2
import sys
import uuid
import time
import dlib
import json
import os, shutil
import numpy as np
import mediapipe as mp

from PIL import Image
from matplotlib import pyplot as plt
from tensorflow.keras.models import load_model
from transformers import T5Tokenizer, T5ForConditionalGeneration

In [ ]:
CONFIG_DIR = os.path.abspath(os.path.join(os.getcwd(), ".."))
if CONFIG_DIR not in sys.path:
    sys.path.insert(0, CONFIG_DIR)

from config import (ROOT, RUNTIME, FRAMES, CROPPED, GRIDS,
                    CNN_MODEL_PATH, PREDICTOR_PATH, LABELS_PATH,
                    SOURCE_VIDEOS, T5_MODEL_DIR, ensure_runtime_dirs)

ensure_runtime_dirs()

In [ ]:
# ROOT = r"D:\lip_codebase_clean"
# RUNTIME = os.path.join(ROOT, "runtime")
# FRAMES     = os.path.join(RUNTIME, "extracted_frames")
# CROPPED    = os.path.join(RUNTIME, "cropped_frames")
# GRIDS      = os.path.join(RUNTIME, "grids")

# for d in [FRAMES, CROPPED, GRIDS]:
#     os.makedirs(d, exist_ok=True)

In [ ]:
def clear_runtime(verbose=True):
    """Empty runtime/ entirely, keeping the runtime folder itself."""
    removed = 0
    for name in os.listdir(RUNTIME):
        path = os.path.join(RUNTIME, name)
        if os.path.isdir(path):
            shutil.rmtree(path)
        else:
            os.remove(path)
        removed += 1
    for d in [FRAMES, CROPPED, GRIDS]:
        os.makedirs(d, exist_ok=True)
    if verbose:
        print(f"removed {removed} items from runtime/")

clear_runtime()

*checks for working camera*

In [ ]:
for i in range(2):
    cap = cv2.VideoCapture(i)
    if cap.isOpened():
        print(f"Camera index {i} is available.")
        cap.release()
    else:
        print(f"Camera index {i} is not available.")

In [ ]:
# cap = cv2.VideoCapture(0)
cap = cv2.VideoCapture(0, cv2.CAP_DSHOW)

In [ ]:
ret, frame = cap.read()

In [ ]:
cap.read()

In [ ]:
cap.release()

In [ ]:
cap.read()

In [ ]:
mp_drawing.DrawingSpec

*test code for hand gesture*

In [ ]:
# Initialize MediaPipe Hands and Drawing modules
mp_hands = mp.solutions.hands
mp_drawing = mp.solutions.drawing_utils

# Initialize video capture
cap = cv2.VideoCapture(0, cv2.CAP_DSHOW)

# Set the resolution to the maximum supported by your camera
cap.set(cv2.CAP_PROP_FRAME_WIDTH, 1280)  # Set width (e.g., 1280 for 720p)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 720)  # Set height (e.g., 720 for 720p)

def is_open_hand(hand_landmarks):
    """ Check if all fingers are extended (open hand) """
    for finger_tip, finger_pip in [
        (mp_hands.HandLandmark.INDEX_FINGER_TIP, mp_hands.HandLandmark.INDEX_FINGER_PIP),
        (mp_hands.HandLandmark.MIDDLE_FINGER_TIP, mp_hands.HandLandmark.MIDDLE_FINGER_PIP),
        (mp_hands.HandLandmark.RING_FINGER_TIP, mp_hands.HandLandmark.RING_FINGER_PIP),
        (mp_hands.HandLandmark.PINKY_TIP, mp_hands.HandLandmark.PINKY_PIP)
    ]:
        if hand_landmarks.landmark[finger_tip].y > hand_landmarks.landmark[finger_pip].y:
            return False  # A finger is not extended
    return True  # All fingers are extended

def is_closed_fist(hand_landmarks):
    """ Check if all fingers are folded (closed fist) """
    for finger_tip, finger_pip in [
        (mp_hands.HandLandmark.INDEX_FINGER_TIP, mp_hands.HandLandmark.INDEX_FINGER_PIP),
        (mp_hands.HandLandmark.MIDDLE_FINGER_TIP, mp_hands.HandLandmark.MIDDLE_FINGER_PIP),
        (mp_hands.HandLandmark.RING_FINGER_TIP, mp_hands.HandLandmark.RING_FINGER_PIP),
        (mp_hands.HandLandmark.PINKY_TIP, mp_hands.HandLandmark.PINKY_PIP)
    ]:
        if hand_landmarks.landmark[finger_tip].y < hand_landmarks.landmark[finger_pip].y:
            return False  # A finger is extended
    return True  # All fingers are folded

with mp_hands.Hands(min_detection_confidence=0.8, min_tracking_confidence=0.5) as hands:
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        # Convert BGR to RGB and flip for mirror effect
        image = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        image = cv2.flip(image, 1)
        image.flags.writeable = False  # Set to False for faster processing

        # Process the image and find hands
        results = hands.process(image)
        image.flags.writeable = True  # Set to True for drawing
        image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)

        # Check if hands are detected
        if results.multi_hand_landmarks:
            for num, hand_landmarks in enumerate(results.multi_hand_landmarks):
                mp_drawing.draw_landmarks(
                    image, hand_landmarks, mp_hands.HAND_CONNECTIONS,
                    mp_drawing.DrawingSpec(color=(121, 22, 76), thickness=2, circle_radius=4),
                    mp_drawing.DrawingSpec(color=(121, 44, 250), thickness=2, circle_radius=2)
                )

                # Gesture recognition logic
                if is_open_hand(hand_landmarks):
                    cv2.putText(image, "Open Hand Detected", (10, 50), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)
                elif is_closed_fist(hand_landmarks):
                    cv2.putText(image, "Closed Fist Detected", (10, 50), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 2)

        # Display the image with results
        cv2.imshow('Hand Gesture Recognition', image)

        if cv2.waitKey(10) & 0xFF == ord('q'):
            break

cap.release()
cv2.destroyAllWindows()

*test code for frame creation with hand gesture*

In [ ]:
# Initialize MediaPipe Hands and Drawing modules
mp_hands = mp.solutions.hands
mp_drawing = mp.solutions.drawing_utils

# Initialize video capture
cap = cv2.VideoCapture(0, cv2.CAP_DSHOW)

# Set the resolution to the maximum supported by your camera
cap.set(cv2.CAP_PROP_FRAME_WIDTH, 1280)  # Set width (e.g., 1280 for 720p)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 720)  # Set height (e.g., 720 for 720p)

# Directory to save the video


# Variables to handle recording
recording = False
captured_frames = []

def is_open_hand(hand_landmarks):
    """ Check if all fingers are extended (open hand) """
    for finger_tip, finger_pip in [
        (mp_hands.HandLandmark.INDEX_FINGER_TIP, mp_hands.HandLandmark.INDEX_FINGER_PIP),
        (mp_hands.HandLandmark.MIDDLE_FINGER_TIP, mp_hands.HandLandmark.MIDDLE_FINGER_PIP),
        (mp_hands.HandLandmark.RING_FINGER_TIP, mp_hands.HandLandmark.RING_FINGER_PIP),
        (mp_hands.HandLandmark.PINKY_TIP, mp_hands.HandLandmark.PINKY_PIP)
    ]:
        if hand_landmarks.landmark[finger_tip].y > hand_landmarks.landmark[finger_pip].y:
            return False  # A finger is not extended
    return True  # All fingers are extended

def is_closed_fist(hand_landmarks):
    """ Check if all fingers are folded (closed fist) """
    for finger_tip, finger_pip in [
        (mp_hands.HandLandmark.INDEX_FINGER_TIP, mp_hands.HandLandmark.INDEX_FINGER_PIP),
        (mp_hands.HandLandmark.MIDDLE_FINGER_TIP, mp_hands.HandLandmark.MIDDLE_FINGER_PIP),
        (mp_hands.HandLandmark.RING_FINGER_TIP, mp_hands.HandLandmark.RING_FINGER_PIP),
        (mp_hands.HandLandmark.PINKY_TIP, mp_hands.HandLandmark.PINKY_PIP)
    ]:
        if hand_landmarks.landmark[finger_tip].y < hand_landmarks.landmark[finger_pip].y:
            return False  # A finger is extended
    return True  # All fingers are folded

with mp_hands.Hands(min_detection_confidence=0.8, min_tracking_confidence=0.5) as hands:
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        image_height, image_width, _ = frame.shape

        # Convert BGR to RGB and flip for mirror effect
        image = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        image = cv2.flip(image, 1)
        image.flags.writeable = False  # Set to False for faster processing

        # Process the image and find hands
        results = hands.process(image)
        image.flags.writeable = True  # Set to True for drawing
        image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)

        # Check if hands are detected
        if results.multi_hand_landmarks:
            for num, hand_landmarks in enumerate(results.multi_hand_landmarks):
                mp_drawing.draw_landmarks(
                    image, hand_landmarks, mp_hands.HAND_CONNECTIONS,
                    mp_drawing.DrawingSpec(color=(121, 22, 76), thickness=2, circle_radius=4),
                    mp_drawing.DrawingSpec(color=(121, 44, 250), thickness=2, circle_radius=2)
                )

                # Gesture recognition logic
                if is_open_hand(hand_landmarks):
                    cv2.putText(image, "Open Hand Detected", (10, 50), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)
                    if not recording:
                        recording = True
                        captured_frames = []
                        print("Recording started")

                elif is_closed_fist(hand_landmarks):
                    cv2.putText(image, "Closed Fist Detected", (10, 50), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 2)
                    if recording:
                        recording = False
                        print(f"Recording stopped — {len(captured_frames)} frames captured")

        # Write frame to the video if recording
        if recording:
            captured_frames.append(frame.copy())
            if len(captured_frames) > 300:
                recording = False
                print("Recording stopped — frame limit reached")

        # Display the image with results
        cv2.imshow('Hand Gesture Recognition', image)

        if cv2.waitKey(10) & 0xFF == ord('q'):
            break

# Release resources
cap.release()
cv2.destroyAllWindows()

In [ ]:
for i, f in enumerate(captured_frames, start=1):
    cv2.imwrite(os.path.join(FRAMES, f"{i:02d}.png"), f)
print(f"wrote {len(captured_frames)} frames to {FRAMES}")

In [ ]:
cap.release()

**demo code**

In [ ]:
# Open and configure the camera before starting the live capture cell
if "cap" in globals() and cap is not None:
    cap.release()

initialization_started = time.perf_counter()

cap = cv2.VideoCapture(0, cv2.CAP_MSMF)
if not cap.isOpened():
    raise RuntimeError("MSMF could not open camera index 0")

cap.set(cv2.CAP_PROP_FRAME_WIDTH, 1920)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 1080)

ret, initialization_frame = cap.read()
if not ret:
    cap.release()
    raise RuntimeError("Camera opened but initialization frame could not be read")

print(f"camera initialization: {time.perf_counter() - initialization_started:.2f}s")
print(
    f"resolution: "
    f"{initialization_frame.shape[1]}x{initialization_frame.shape[0]}"
)
print(f"reported fps: {cap.get(cv2.CAP_PROP_FPS):.1f}")
print("camera ready")

In [ ]:
cell_started = time.perf_counter()

clear_runtime()

# Initialize MediaPipe Hands and Drawing modules
mp_hands = mp.solutions.hands
mp_drawing = mp.solutions.drawing_utils

# Reuse the MSMF camera opened by the initialization cell
if "cap" not in globals() or cap is None or not cap.isOpened():
    raise RuntimeError("Run the camera initialization cell first")

# Variables to handle recording
recording = False
finished = False          # set once one full open -> fist cycle completes
captured_frames = []
t_start = None
elapsed = 0.0

def is_open_hand(hand_landmarks):
    """ Check if all fingers are extended (open hand) """
    for finger_tip, finger_pip in [
        (mp_hands.HandLandmark.INDEX_FINGER_TIP, mp_hands.HandLandmark.INDEX_FINGER_PIP),
        (mp_hands.HandLandmark.MIDDLE_FINGER_TIP, mp_hands.HandLandmark.MIDDLE_FINGER_PIP),
        (mp_hands.HandLandmark.RING_FINGER_TIP, mp_hands.HandLandmark.RING_FINGER_PIP),
        (mp_hands.HandLandmark.PINKY_TIP, mp_hands.HandLandmark.PINKY_PIP)
    ]:
        if hand_landmarks.landmark[finger_tip].y > hand_landmarks.landmark[finger_pip].y:
            return False
    return True

def is_closed_fist(hand_landmarks):
    """ Check if all fingers are folded (closed fist) """
    for finger_tip, finger_pip in [
        (mp_hands.HandLandmark.INDEX_FINGER_TIP, mp_hands.HandLandmark.INDEX_FINGER_PIP),
        (mp_hands.HandLandmark.MIDDLE_FINGER_TIP, mp_hands.HandLandmark.MIDDLE_FINGER_PIP),
        (mp_hands.HandLandmark.RING_FINGER_TIP, mp_hands.HandLandmark.RING_FINGER_PIP),
        (mp_hands.HandLandmark.PINKY_TIP, mp_hands.HandLandmark.PINKY_PIP)
    ]:
        if hand_landmarks.landmark[finger_tip].y < hand_landmarks.landmark[finger_pip].y:
            return False
    return True

preview_reported = False
mediapipe_started = time.perf_counter()

with mp_hands.Hands(min_detection_confidence=0.8, min_tracking_confidence=0.5) as hands:
    print(f"MediaPipe initialization: {time.perf_counter() - mediapipe_started:.2f}s")
    while cap.isOpened() and not finished:
        if not preview_reported:
            first_loop_started = time.perf_counter()
        ret, frame = cap.read()
        if not ret:
            break

        image = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        image = cv2.flip(image, 1)
        image.flags.writeable = False
        results = hands.process(image)
        image.flags.writeable = True
        image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)

        if results.multi_hand_landmarks:
            for num, hand_landmarks in enumerate(results.multi_hand_landmarks):
                mp_drawing.draw_landmarks(
                    image, hand_landmarks, mp_hands.HAND_CONNECTIONS,
                    mp_drawing.DrawingSpec(color=(121, 22, 76), thickness=2, circle_radius=4),
                    mp_drawing.DrawingSpec(color=(121, 44, 250), thickness=2, circle_radius=2)
                )

                if is_open_hand(hand_landmarks):
                    cv2.putText(image, "Open Hand Detected", (10, 50),
                                cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)
                    if not recording:
                        recording = True
                        captured_frames = []
                        t_start = time.time()
                        print("Recording started - speak now")

                elif is_closed_fist(hand_landmarks):
                    cv2.putText(image, "Closed Fist Detected", (10, 50),
                                cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 2)
                    if recording:
                        recording = False
                        finished = True
                        elapsed = time.time() - t_start
                        print(f"Recording stopped - {len(captured_frames)} frames "
                              f"in {elapsed:.2f}s ({len(captured_frames)/elapsed:.1f} fps)")

        if recording:
            captured_frames.append(frame.copy())
            elapsed = time.time() - t_start
            if len(captured_frames) >= 300:
                recording = False
                finished = True
                print(f"Recording stopped - frame limit reached at {elapsed:.2f}s")

        # timer and frame count overlay
        if recording:
            cv2.putText(image, f"REC  {elapsed:5.2f}s   {len(captured_frames)} frames",
                        (10, 100), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 0, 255), 2)
            cv2.circle(image, (image.shape[1] - 40, 40), 12, (0, 0, 255), -1)
        elif not finished:
            cv2.putText(image, "Open hand to start", (10, 100),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.9, (200, 200, 200), 2)

        cv2.imshow("Hand Gesture Recognition", image)
        key = cv2.waitKey(1) & 0xFF

        if not preview_reported:
            first_loop_elapsed = time.perf_counter() - first_loop_started
            preview_elapsed = time.perf_counter() - cell_started

            print(f"first processing loop: {first_loop_elapsed:.2f}s")
            print(f"time to first preview: {preview_elapsed:.2f}s")

            preview_reported = True

        if key == ord("q"):
            if recording:
                captured_frames = []
                recording = False
                print("recording cancelled")
            break

cv2.destroyAllWindows()

# save frames to disk and release memory
if captured_frames:
    for i, f in enumerate(captured_frames, start=1):
        cv2.imwrite(os.path.join(FRAMES, f"{i:02d}.png"), f)
    print(f"wrote {len(captured_frames)} frames to {FRAMES}")
    print(f"duration {elapsed:.2f}s")
    captured_frames = []
else:
    print("no frames captured")

In [ ]:
# # Run only when the complete camera session is finished
# if "cap" in globals() and cap is not None and cap.isOpened():
#     cap.release()

# cv2.destroyAllWindows()
# print("camera released")

*further frame processing*

In [ ]:
TARGET_FRAMES = 60

def normalise_extracted_frames(frames_dir=FRAMES, target=TARGET_FRAMES):
    """Trim or pad the captured frames to exactly `target`, in memory,
    then rewrite frames_dir as 01..target. Same rule as the training
    pipeline: pad with the last frame, trim 80% from the end."""
    files = [f for f in os.listdir(frames_dir) if f.endswith(".png")]
    files.sort(key=lambda f: int(os.path.splitext(f)[0]))

    if not files:
        raise RuntimeError(f"no frames found in {frames_dir}")

    frames = [cv2.imread(os.path.join(frames_dir, f)) for f in files]
    source_count = len(frames)
    padded = trimmed_start = trimmed_end = 0

    if source_count < target:
        padded = target - source_count
        frames = frames + [frames[-1]] * padded
    elif source_count > target:
        excess = source_count - target
        trimmed_end = int(excess * 0.8)
        trimmed_start = excess - trimmed_end
        frames = frames[trimmed_start:source_count - trimmed_end]

    for f in files:
        os.remove(os.path.join(frames_dir, f))
    for i, frame in enumerate(frames, start=1):
        cv2.imwrite(os.path.join(frames_dir, f"{i:02d}.png"), frame)

    print(f"source frames : {source_count}")
    print(f"padded        : {padded}")
    print(f"trimmed start : {trimmed_start}")
    print(f"trimmed end   : {trimmed_end}")
    print(f"written       : {len(frames)} to {frames_dir}")
    return dict(source=source_count, padded=padded,
                trimmed_start=trimmed_start, trimmed_end=trimmed_end)

frame_stats = normalise_extracted_frames()

In [ ]:
PREDICTOR_PATH = os.path.join(ROOT, "models", "shape_predictor_68_face_landmarks.dat")
LIP_HEIGHT, LIP_WIDTH = 80, 112

_detector = dlib.get_frontal_face_detector()
_predictor = dlib.shape_predictor(PREDICTOR_PATH)


def crop_lip(frame):
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    faces = _detector(gray)
    if not faces:
        return None
    landmarks = _predictor(gray, faces[0])
    mouth = np.array([(landmarks.part(n).x, landmarks.part(n).y) for n in range(48, 68)])
    x, y, w, h = cv2.boundingRect(mouth)

    pad_left   = max((LIP_WIDTH - w) // 2, 0)
    pad_right  = max((LIP_WIDTH - w) - pad_left, 0)
    pad_top    = max((LIP_HEIGHT - h) // 2, 0)
    pad_bottom = max((LIP_HEIGHT - h) - pad_top, 0)

    pad_left   = min(pad_left, x)
    pad_right  = min(pad_right, frame.shape[1] - (x + w))
    pad_top    = min(pad_top, y)
    pad_bottom = min(pad_bottom, frame.shape[0] - (y + h))

    lip = frame[y - pad_top:y + h + pad_bottom, x - pad_left:x + w + pad_right]
    if lip.size == 0:
        return None
    return cv2.resize(lip, (LIP_WIDTH, LIP_HEIGHT))


def fill_gaps(crops):
    n = len(crops)
    if not any(c is not None for c in crops):
        return None, None
    filled, records, i = list(crops), [], 0
    while i < n:
        if filled[i] is not None:
            i += 1
            continue
        start = i
        while i < n and crops[i] is None:
            i += 1
        end, length = i - 1, i - start
        before = start - 1 if start > 0 else None
        after  = i if i < n else None
        if before is None:
            for k in range(start, end + 1):
                filled[k] = crops[after]; records.append((k + 1, after + 1))
        elif after is None:
            for k in range(start, end + 1):
                filled[k] = crops[before]; records.append((k + 1, before + 1))
        else:
            first_half = (length + 1) // 2
            for offset, k in enumerate(range(start, end + 1)):
                src = before if offset < first_half else after
                filled[k] = crops[src]; records.append((k + 1, src + 1))
    return filled, records


def crop_extracted_frames(frames_dir=FRAMES, cropped_dir=CROPPED):
    for f in os.listdir(cropped_dir):
        os.remove(os.path.join(cropped_dir, f))

    files = sorted(f for f in os.listdir(frames_dir) if f.endswith(".png"))
    crops = []
    for f in files:
        frame = cv2.imread(os.path.join(frames_dir, f))
        crops.append(None if frame is None else crop_lip(frame))

    failed = [i + 1 for i, c in enumerate(crops) if c is None]
    filled, records = fill_gaps(crops)

    if filled is None:
        raise RuntimeError("dlib detected no face in any frame - recapture needed")

    for i, c in enumerate(filled, start=1):
        cv2.imwrite(os.path.join(cropped_dir, f"{i:02d}.png"), c)

    print(f"frames in     : {len(crops)}")
    print(f"detected      : {len(crops) - len(failed)}")
    print(f"failed frames : {failed if failed else 'none'}")
    print(f"filled from   : {records if records else 'none'}")
    print(f"written       : {len(filled)} to {cropped_dir}")
    return dict(total=len(crops), detected=len(crops) - len(failed),
                failed=failed, fills=records)

crop_stats = crop_extracted_frames()

In [ ]:
def measure_mouth_boxes(frames_dir=FRAMES):
    files = sorted(f for f in os.listdir(frames_dir) if f.endswith(".png"))
    widths, heights = [], []
    for f in files:
        frame = cv2.imread(os.path.join(frames_dir, f))
        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        faces = _detector(gray)
        if not faces:
            continue
        landmarks = _predictor(gray, faces[0])
        mouth = np.array([(landmarks.part(n).x, landmarks.part(n).y) for n in range(48, 68)])
        x, y, w, h = cv2.boundingRect(mouth)
        widths.append(w)
        heights.append(h)

    print(f"frames measured : {len(widths)}")
    print(f"source frame    : {frame.shape[1]}x{frame.shape[0]}")
    print(f"mouth width  min/mean/max : {min(widths)} / {sum(widths)/len(widths):.1f} / {max(widths)}")
    print(f"mouth height min/mean/max : {min(heights)} / {sum(heights)/len(heights):.1f} / {max(heights)}")
    print(f"crop target     : {LIP_WIDTH}x{LIP_HEIGHT}")
    return widths, heights

box_w, box_h = measure_mouth_boxes()

In [ ]:
# SOURCE_VIDEOS = r"D:\lip_codebase_old\Data for lip reading model\Videos of 10 selected words named"

# def measure_source_videos(root=SOURCE_VIDEOS, sets=None, words=None, frames_per_video=5):
#     """Measure dlib mouth bounding boxes in the original recordings."""
#     if sets is None:
#         sets = list(range(1, 61))
#     if words is None:
#         words = ['bat', 'cup', 'drop', 'eat', 'fish', 'hot', 'jump', 'milk', 'pen', 'red']

#     widths, heights, shapes = [], [], set()
#     missing, no_face = 0, 0

#     for n in sets:
#         for word in words:
#             path = os.path.join(root, f"set_{n:02d}", f"{word}.mp4")
#             if not os.path.exists(path):
#                 missing += 1
#                 continue
#             cap_v = cv2.VideoCapture(path)
#             total = int(cap_v.get(cv2.CAP_PROP_FRAME_COUNT))
#             if total <= 0:
#                 cap_v.release()
#                 continue
#             picks = [int(total * p) for p in (0.3, 0.4, 0.5, 0.6, 0.7)][:frames_per_video]
#             for idx in picks:
#                 cap_v.set(cv2.CAP_PROP_POS_FRAMES, idx)
#                 ok, frame = cap_v.read()
#                 if not ok:
#                     continue
#                 shapes.add((frame.shape[1], frame.shape[0]))
#                 gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
#                 faces = _detector(gray)
#                 if not faces:
#                     no_face += 1
#                     continue
#                 landmarks = _predictor(gray, faces[0])
#                 mouth = np.array([(landmarks.part(k).x, landmarks.part(k).y)
#                                   for k in range(48, 68)])
#                 x, y, w, h = cv2.boundingRect(mouth)
#                 widths.append(w)
#                 heights.append(h)
#             cap_v.release()

#     print(f"videos missing   : {missing}")
#     print(f"frames measured  : {len(widths)}")
#     print(f"frames no face   : {no_face}")
#     print(f"source shapes    : {shapes}")
#     print(f"mouth width  min/mean/max : {min(widths)} / {sum(widths)/len(widths):.1f} / {max(widths)}")
#     print(f"mouth height min/mean/max : {min(heights)} / {sum(heights)/len(heights):.1f} / {max(heights)}")
#     print(f"crop target      : {LIP_WIDTH}x{LIP_HEIGHT}")
#     return widths, heights

# src_w, src_h = measure_source_videos()

In [ ]:
ROWS, COLS = 10, 6
GRID_NAME = "grid.png"
GRID_RESIZED_NAME = "grid_resized.png"
NEW_SIZE = (224, 224)


def build_grid(cropped_dir=CROPPED, grids_dir=GRIDS):
    frames = sorted(f for f in os.listdir(cropped_dir) if f.endswith(".png"))
    if len(frames) != ROWS * COLS:
        raise RuntimeError(f"expected {ROWS * COLS} crops, found {len(frames)}")

    first = cv2.imread(os.path.join(cropped_dir, frames[0]))
    fh, fw, ch = first.shape
    grid = np.zeros((fh * ROWS, fw * COLS, ch), dtype=np.uint8)

    for idx, f in enumerate(frames):
        img = cv2.imread(os.path.join(cropped_dir, f))
        if img is None:
            raise RuntimeError(f"could not read {f}")
        r, c = idx // COLS, idx % COLS
        grid[r * fh:(r + 1) * fh, c * fw:(c + 1) * fw] = img

    grid_path = os.path.join(grids_dir, GRID_NAME)
    cv2.imwrite(grid_path, grid)

    resized_path = os.path.join(grids_dir, GRID_RESIZED_NAME)
    with Image.open(grid_path) as img:
        img.resize(NEW_SIZE, Image.LANCZOS).save(resized_path)

    print(f"cell size    : {fw}x{fh}")
    print(f"grid         : {grid.shape[1]}x{grid.shape[0]} ({ROWS} rows x {COLS} cols)")
    print(f"resized      : {NEW_SIZE[0]}x{NEW_SIZE[1]}")
    print(f"written      : {grids_dir}")
    return resized_path

grid_path = build_grid()

In [ ]:
MODEL_PATH = os.path.join(ROOT, "models", "model_rebuilt_data.h5")
LABELS_PATH = os.path.join(ROOT, "models", "class_labels_cl10.json")

lip_model = None
class_labels = None


def load_lip_model():
    global lip_model, class_labels
    lip_model = load_model(MODEL_PATH)
    with open(LABELS_PATH) as f:
        class_labels = json.load(f)
    print(f"model  : {os.path.basename(MODEL_PATH)}")
    print(f"params : {lip_model.count_params():,}")
    print(f"input  : {lip_model.input_shape}")
    print(f"labels : {list(class_labels.values())}")


def predict_word(grid_image_path=None, top_k=3):
    if lip_model is None:
        raise RuntimeError("run load_lip_model() first")
    if grid_image_path is None:
        grid_image_path = os.path.join(GRIDS, GRID_RESIZED_NAME)

    with Image.open(grid_image_path) as img:
        arr = np.array(img.convert("RGB")) / 255.0
    arr = np.expand_dims(arr, axis=0)

    probs = lip_model.predict(arr, verbose=0)[0]
    order = np.argsort(probs)[::-1]

    print(f"predicted : {class_labels[str(order[0])]}  ({probs[order[0]]:.4f})")
    print("top", top_k, ":")
    for i in order[:top_k]:
        print(f"  {class_labels[str(i)]:6s} {probs[i]:.4f}")
    return class_labels[str(order[0])], probs

load_lip_model()

In [ ]:
predicted_word, probabilities = predict_word()

In [ ]:
t5_tokenizer = None
t5_model = None


def load_t5_model():
    global t5_tokenizer, t5_model
    t5_tokenizer = T5Tokenizer.from_pretrained(T5_MODEL_DIR)
    t5_model = T5ForConditionalGeneration.from_pretrained(T5_MODEL_DIR)
    print(f"tokenizer : {type(t5_tokenizer).__name__}")
    print(f"model     : {type(t5_model).__name__}")
    print(f"loaded    : {T5_MODEL_DIR}")


def generate_sentence(word, max_length=20, top_k=50, top_p=0.9, temperature=0.6):
    """Generate a sentence containing the predicted word.

    Prompt format and sampling settings are taken from the original
    integration notebook so that generation matches what the model
    was fine-tuned on."""
    if t5_model is None:
        raise RuntimeError("run load_t5_model() first")

    input_text = f"Generate a sentence for {word}:"
    input_ids = t5_tokenizer(input_text, return_tensors="pt").input_ids

    outputs = t5_model.generate(
        input_ids,
        max_length=max_length,
        do_sample=True,
        top_k=top_k,
        top_p=top_p,
        temperature=temperature,
        num_return_sequences=1
    )

    sentence = t5_tokenizer.decode(outputs[0], skip_special_tokens=True)
    print(f"prompt    : {input_text}")
    print(f"sentence  : {sentence}")
    return sentence

load_t5_model()

In [ ]:
sentence = generate_sentence(predicted_word)

In [ ]:
# sentence = generate_sentence("hot")